# Bistable ODE Simulation
This notebook simulates the bistable ODE:


where $r\sim \mathcal{U}(0.8, 1.2)$ and $y_0 \sim \mathcal{U}(0,4)$.


# Bayesian Inference for the Bistable ODE

Uses the trained PINN surrogate (from `train.py`) as a cheap forward model inside an MCMC loop.




**Problem setup**
- Forward model: 
$$
\frac{dy}{dt} = -r(y - 1) (2 - y) (y - 3) \\
y(0) = y_0
$$

- Unknown parameters: $y_0 \in (0, 4)$, $r \in (0.8, 1.2)$

- Observation model: 
$$
y_{\text{obs},i} = y(t_i; y_0, r) + \varepsilon_i, \quad \varepsilon_i \sim \mathcal{N}(0, \sigma^2)
$$

- Priors:
$$
y_0 \sim \mathcal{U}(0, 4) \\
r \sim \mathcal{U}(0.8, 1.2) \\
\sigma \sim \mathcal{U}(0.0001, 1.0)
$$

- Sampler: `emcee` ensemble MCMC

In [ ]:
%matplotlib widget
%reload_ext autoreload
%autoreload 2

import arviz as az
import emcee
import matplotlib.pyplot as plt
import numpy as np
import torch

from examples.equations.bistable_ode.ode import reference_solution
from measure_uq.models import PINN

rng = np.random.default_rng(seed=42)

## Generate synthetic observations

In [ ]:
# true parameter values used to generate synthetic data
true_y0 = 1.5
true_r = 1
true_sigma = 0.05

# observation settings
n_obs = 200
T = 8.0

t_obs = np.linspace(0.2, T, n_obs)
p_true = np.array([true_y0, true_r])
y_true = reference_solution(t_obs, p_true)
y_obs = y_true + rng.normal(0.0, true_sigma, size=n_obs)

np.savez("data/observations.npz", t=t_obs, y_obs=y_obs, y_true=y_true)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(t_obs, y_true, label="true solution")
ax.scatter(t_obs, y_obs, s=20, zorder=5, label="observations")
ax.set_xlabel("$t$")
ax.set_ylabel("$y$")
ax.legend()
ax.set_title(f"Synthetic data  ($y_0={true_y0}$, $r={true_r}$, $\\sigma={true_sigma}$)")
ax.grid()
plt.tight_layout()
plt.show()

## Bayesian inference setup

In [ ]:
checkpoint = torch.load(
    "data/best_model_pinn.pickle", map_location="cpu", weights_only=False
)
model = PINN(network_builder=checkpoint["network_builder"])
model.load_state_dict(checkpoint["state_dict"])
model.eval()


def predict(t: np.ndarray, y0: float, r: float) -> np.ndarray:
    with torch.no_grad():
        x = torch.tensor(t.reshape(-1, 1), dtype=torch.float32)
        p = torch.tensor([[y0, r]], dtype=torch.float32)
        _, y = model(x, p)
    return y.numpy().flatten()


def log_prior(theta: np.ndarray) -> float:
    """Uniform prior over the training domain."""
    y0, r, sigma = theta
    if 0.0 < y0 < 4.0 and 0.8 < r < 1.2 and 0.0001 < sigma < 1.0:
        return 0.0
    return -np.inf


def log_likelihood(theta: np.ndarray) -> float:
    """Gaussian log-likelihood with constant noise sigma."""
    y0, r, sigma = theta
    y_pred = predict(t_obs, y0, r)
    if not np.all(np.isfinite(y_pred)):
        return -np.inf
    return -0.5 * np.sum(((y_obs - y_pred) / sigma) ** 2) - n_obs * np.log(sigma)


def log_prob(theta: np.ndarray) -> float:
    """Unnormalized log-posterior."""
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood(theta)

## Run MCMC with emcee

In [ ]:
# MCMC settings
nwalkers = 32
nsteps = 3000
discard = 500
thin = 10

ndim = 3  # (y0, r, sigma)

# Initialise in the correct basin (y0 < 2 → converges to y=1).
# Looking at the data tells us which basin we are in.
p0 = np.column_stack(
    [
        rng.uniform(0.2, 1.9, nwalkers),  # y0 — restricted to y<2 basin
        rng.uniform(0.85, 1.15, nwalkers),  # r
        rng.uniform(0.02, 0.15, nwalkers),  # sigma
    ]
)

sampler = emcee.EnsembleSampler(nwalkers, ndim, log_prob)
_ = sampler.run_mcmc(p0, nsteps, progress=True)

In [ ]:
idata = az.from_emcee(sampler, var_names=["y0", "r", "sigma"])
idata_burned = idata.isel(draw=slice(discard, None, thin))

az.summary(idata_burned)

In [ ]:
az.plot_trace(idata, figure_kwargs={"figsize": (7, 4)})
plt.tight_layout()
plt.show()

In [ ]:
az.plot_pair(idata_burned, marginal=True, figure_kwargs={"figsize": (7, 4)})
plt.tight_layout()
plt.show()

In [ ]:
post = az.extract(idata_burned)
t_fine = np.linspace(0.0, T, 200)
n_draws = 1000
idx = np.random.default_rng(0).integers(0, post.sizes["sample"], n_draws)

# Draw from the full generative model: y_pred ~ N(f(t; y0, r), sigma^2)
ppc = np.stack(
    [
        predict(t_fine, float(post["y0"][i]), float(post["r"][i]))
        + rng.normal(0.0, float(post["sigma"][i]), size=len(t_fine))
        for i in idx
    ]
)  # shape (n_draws, len(t_fine))

q05, q50, q95 = np.percentile(ppc, [5, 50, 95], axis=0)

fig, ax = plt.subplots(figsize=(7, 4))
ax.fill_between(t_fine, q05, q95, color="C0", alpha=0.25, label="90% predictive CI")
ax.plot(t_fine, q50, color="C0", lw=2, label="posterior median")
ax.plot(
    t_fine, reference_solution(t_fine, p_true), "k--", lw=1.5, label="true solution"
)
ax.scatter(t_obs, y_obs, s=25, zorder=5, color="C1", label="observations")
ax.set_xlabel("$t$")
ax.set_ylabel("$y$")
ax.legend()
ax.set_title("Posterior predictive check (90% credible interval)")
plt.tight_layout()
plt.show()

In [ ]:
t_check = np.linspace(0.2, T, 200)
y_pinn = predict(t_check, true_y0, true_r)
y_ref = reference_solution(t_check, p_true)

fig, axes = plt.subplots(1, 2, figsize=(7, 4))
axes[0].plot(t_check, y_ref, "k-", lw=1.5, label="reference")
axes[0].plot(t_check, y_pinn, "C0--", lw=1.5, label="PINN")
axes[0].set_xlabel("$t$")
axes[0].set_ylabel("$y$")
axes[0].legend()
axes[0].set_title("PINN vs reference at true parameters")
axes[0].grid()

axes[1].plot(t_check, y_pinn - y_ref, "C1")
axes[1].set_xlabel("$t$")
axes[1].set_ylabel("error")
axes[1].set_title(f"Pointwise error  (max={np.abs(y_pinn - y_ref).max():.4f})")
axes[1].grid()

plt.tight_layout()
plt.show()